In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import mean_squared_log_error
import time

DATA_DIR = "../data"
train = pd.read_parquet(f"{DATA_DIR}/train_processed_v3.parquet")

categorical_cols = ["store_nbr", "family", "city", "state", "type", "cluster"]
for col in categorical_cols:
    train[col] = train[col].astype("category")

exclude_cols = ["id", "date", "sales", "sales_log", "is_train"]
feature_cols = [col for col in train.columns if col not in exclude_cols]

val_start = train["date"].max() - pd.Timedelta(days=15)
train_honest = train[train["date"] < val_start].copy()
val_true = train[train["date"] >= val_start][["id", "date", "store_nbr", "family", "sales"]].copy()

X_train = train_honest[feature_cols]
y_train = train_honest["sales_log"]
X_val = train[train["date"] >= val_start][feature_cols]

results = {}
print("Setup complete. Train rows:", len(X_train), "Val rows:", len(X_val))

Setup complete. Train rows: 2979504 Val rows: 28512


In [2]:
def evaluate_model(name, model, X_val_input, is_log_target=True):
    start = time.time()
    pred = model.predict(X_val_input)
    if is_log_target:
        pred = np.expm1(pred)
    pred = np.clip(pred, 0, None)

    val_check = val_true.copy()
    val_check["pred"] = pred
    score = np.sqrt(mean_squared_log_error(val_check["sales"], val_check["pred"]))

    elapsed = time.time() - start
    results[name] = score
    print(f"{name}: RMSLE = {score:.4f} (predict time: {elapsed:.1f}s)")
    return score


best_params = {
    "learning_rate": 0.0404, "num_leaves": 131, "max_depth": 10,
    "min_child_samples": 43, "subsample": 0.8245, "colsample_bytree": 0.9222,
    "reg_alpha": 0.0177, "reg_lambda": 2.2852, "random_state": 42, "verbosity": -1
}

t0 = time.time()
lgbm_model = lgb.LGBMRegressor(n_estimators=486, **best_params)
lgbm_model.fit(X_train, y_train)
print(f"LightGBM trained in {time.time()-t0:.1f}s")

evaluate_model("LightGBM", lgbm_model, X_val)

LightGBM trained in 19.5s
LightGBM: RMSLE = 0.3802 (predict time: 0.2s)


np.float64(0.3801881885732322)

In [3]:
def recursive_predict(model, model_name, predict_fn=None, encode_categorical=False):
    """
    predict_fn: custom function(model, X) -> predictions in log scale.
                If None, uses model.predict(X) directly.
    encode_categorical: if True, converts category columns to codes before predicting
                         (needed for XGBoost, sklearn models).
    """
    history = train_honest[["date", "store_nbr", "family", "sales", "onpromotion"]].copy()
    test_onpromo_local = train[train["date"] >= val_start][["date", "store_nbr", "family", "onpromotion"]].copy()

    static_cols_local = [c for c in feature_cols if c not in [
        "lag_7", "lag_14", "lag_28", "lag_364",
        "rolling_mean_7", "rolling_std_7", "rolling_mean_28", "rolling_std_28",
        "promo_typical", "promo_relative", "onpromotion", "store_nbr", "family"
    ]]
    static_feats = train[train["date"] >= val_start][
        static_cols_local + ["date", "store_nbr", "family", "id"]
    ].drop_duplicates()

    working = pd.concat([
        history,
        val_true[["date", "store_nbr", "family"]].merge(test_onpromo_local, on=["date", "store_nbr", "family"], how="left").assign(sales=np.nan)
    ], axis=0, ignore_index=True)

    val_dates_local = sorted(val_true["date"].unique())
    predictions = {}

    for current_date in val_dates_local:
        working = working.sort_values(["store_nbr", "family", "date"])

        for lag in [7, 14, 28, 364]:
            working[f"lag_{lag}"] = working.groupby(["store_nbr", "family"])["sales"].shift(lag)

        for window in [7, 28]:
            working[f"rolling_mean_{window}"] = (
                working.groupby(["store_nbr", "family"])["sales"]
                .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
            )
            working[f"rolling_std_{window}"] = (
                working.groupby(["store_nbr", "family"])["sales"]
                .transform(lambda x: x.shift(1).rolling(window, min_periods=1).std())
            )

        working["promo_typical"] = (
            working.groupby(["store_nbr", "family"])["onpromotion"]
            .transform(lambda x: x.shift(1).rolling(90, min_periods=1).mean())
        )
        working["promo_relative"] = working["onpromotion"] / (working["promo_typical"] + 1)

        day_mask = (working["date"] == current_date) & (working["sales"].isna())
        day_rows = working[day_mask].copy()
        day_rows = day_rows.merge(static_feats, on=["date", "store_nbr", "family"], how="left")

        for col in categorical_cols:
            day_rows[col] = day_rows[col].astype("category")

        X_day = day_rows[feature_cols].copy()
        if encode_categorical:
            for col in categorical_cols:
                X_day[col] = X_day[col].cat.codes

        if predict_fn:
            day_pred_log = predict_fn(model, X_day)
        else:
            day_pred_log = model.predict(X_day)
        day_pred = np.clip(np.expm1(day_pred_log), 0, None)

        for idx, (_, row) in enumerate(day_rows.iterrows()):
            predictions[row["id"]] = day_pred[idx]

        update_mask = (working["date"] == current_date)
        working.loc[update_mask, "sales"] = working.loc[update_mask].merge(
            day_rows[["store_nbr", "family", "date"]].assign(pred=day_pred),
            on=["store_nbr", "family", "date"], how="left"
        )["pred"].values

    val_check = val_true.copy()
    val_check["pred"] = val_check["id"].map(predictions).clip(lower=0)
    score = np.sqrt(mean_squared_log_error(val_check["sales"], val_check["pred"]))
    results[model_name] = score
    print(f"{model_name}: RMSLE = {score:.4f}")
    return predictions, score

In [4]:
lgbm_preds, lgbm_score = recursive_predict(lgbm_model, "LightGBM (recursive)")

LightGBM (recursive): RMSLE = 0.4060


In [5]:
X_train_encoded = X_train.copy()
for col in categorical_cols:
    X_train_encoded[col] = X_train_encoded[col].cat.codes

t0 = time.time()
xgb_model = xgb.XGBRegressor(
    n_estimators=486, learning_rate=0.0404, max_depth=10,
    min_child_weight=43, subsample=0.8245, colsample_bytree=0.9222,
    reg_alpha=0.0177, reg_lambda=2.2852, random_state=42, verbosity=0
)
xgb_model.fit(X_train_encoded, y_train)
print(f"XGBoost trained in {time.time()-t0:.1f}s")

xgb_preds, xgb_score = recursive_predict(xgb_model, "XGBoost (recursive)", encode_categorical=True)

XGBoost trained in 82.2s
XGBoost (recursive): RMSLE = 0.4009


In [6]:
all_models_results = {"LightGBM": lgbm_score, "XGBoost": xgb_score}
all_predictions = {"LightGBM": lgbm_preds, "XGBoost": xgb_preds}

def run_full_test(name, train_fn, predict_fn=None, encode_categorical=False):
    print(f"\n{'='*50}\nStarting: {name}\n{'='*50}")
    t0 = time.time()
    model = train_fn()
    print(f"{name} trained in {time.time()-t0:.1f}s")
    preds, score = recursive_predict(model, name, predict_fn=predict_fn, encode_categorical=encode_categorical)
    all_models_results[name] = score
    all_predictions[name] = preds
    return model, preds

cat_idx = [X_train.columns.get_loc(c) for c in categorical_cols]
catboost_model, _ = run_full_test(
    "CatBoost",
    lambda: CatBoostRegressor(iterations=486, learning_rate=0.0404, depth=10,
                               l2_leaf_reg=2.2852, random_state=42, verbose=False,
                               cat_features=cat_idx).fit(X_train, y_train)
)

hgb_model, _ = run_full_test(
    "HistGradientBoosting",
    lambda: HistGradientBoostingRegressor(max_iter=486, learning_rate=0.0404, max_depth=10, random_state=42).fit(X_train_encoded, y_train),
    encode_categorical=True
)

rf_model, _ = run_full_test(
    "Random Forest",
    lambda: RandomForestRegressor(n_estimators=200, max_depth=15, n_jobs=-1, random_state=42).fit(X_train_encoded, y_train),
    encode_categorical=True
)

et_model, _ = run_full_test(
    "Extra Trees",
    lambda: ExtraTreesRegressor(n_estimators=200, max_depth=15, n_jobs=-1, random_state=42).fit(X_train_encoded, y_train),
    encode_categorical=True
)


Starting: CatBoost
CatBoost trained in 1109.7s
CatBoost: RMSLE = 0.4292

Starting: HistGradientBoosting
HistGradientBoosting trained in 39.6s
HistGradientBoosting: RMSLE = 0.4302

Starting: Random Forest
Random Forest trained in 891.0s
Random Forest: RMSLE = 0.4497

Starting: Extra Trees


KeyboardInterrupt: 

In [ ]:
print(f"\n{'='*50}\nINDIVIDUAL MODEL RESULTS\n{'='*50}")
sorted_results = sorted(all_models_results.items(), key=lambda x: x[1])
for name, score in sorted_results:
    print(f"{name}: {score:.4f}")

def ensemble_score(model_names):
    val_check = val_true.copy()
    preds_matrix = pd.DataFrame({name: val_check["id"].map(all_predictions[name]) for name in model_names})
    val_check["pred"] = preds_matrix.mean(axis=1).clip(lower=0)
    return np.sqrt(mean_squared_log_error(val_check["sales"], val_check["pred"]))

best_2 = [x[0] for x in sorted_results[:2]]
best_3 = [x[0] for x in sorted_results[:3]]
all_available = list(all_predictions.keys())

print(f"\nTop 2 ensemble ({best_2}): {ensemble_score(best_2):.4f}")
print(f"Top 3 ensemble ({best_3}): {ensemble_score(best_3):.4f}")
print(f"All {len(all_available)} models ensemble: {ensemble_score(all_available):.4f}")

In [ ]:
from itertools import combinations

print(f"\n{'='*50}\nALL 2-MODEL COMBINATIONS\n{'='*50}")
model_names = list(all_predictions.keys())
two_combos = []
for combo in combinations(model_names, 2):
    score = ensemble_score(list(combo))
    two_combos.append((combo, score))
    print(f"{combo}: {score:.4f}")

print(f"\n{'='*50}\nALL 3-MODEL COMBINATIONS\n{'='*50}")
three_combos = []
for combo in combinations(model_names, 3):
    score = ensemble_score(list(combo))
    three_combos.append((combo, score))
    print(f"{combo}: {score:.4f}")

print(f"\n{'='*50}\nBEST OVERALL\n{'='*50}")
all_combos = two_combos + three_combos
all_combos.sort(key=lambda x: x[1])
for combo, score in all_combos[:5]:
    print(f"{combo}: {score:.4f}")

In [ ]:
last_date = train["date"].max()
lookback_start = last_date - pd.Timedelta(days=365)

recent_sales = train[train["date"] >= lookback_start].groupby(["store_nbr", "family"])["sales"].sum()
zero_force_combos = recent_sales[recent_sales == 0].index

print("Number of store+family combinations to zero-force:", len(zero_force_combos))

def ensemble_score_zero_forced(model_names, zero_combos):
    val_check = val_true.copy()
    preds_matrix = pd.DataFrame({name: val_check["id"].map(all_predictions[name]) for name in model_names})
    val_check["pred"] = preds_matrix.mean(axis=1).clip(lower=0)
    idx = val_check.set_index(["store_nbr", "family"]).index
    val_check.loc[idx.isin(zero_combos), "pred"] = 0.0
    return np.sqrt(mean_squared_log_error(val_check["sales"], val_check["pred"]))

score_with_zf = ensemble_score_zero_forced(["LightGBM", "XGBoost"], zero_force_combos)
print(f"LightGBM + XGBoost ensemble WITH zero-forcing: {score_with_zf:.4f}")

In [ ]:
best_weight_score = None
best_weight = None

for w in np.arange(0.1, 1.0, 0.05):
    val_check = val_true.copy()
    lgbm_p = val_check["id"].map(all_predictions["LightGBM"])
    xgb_p = val_check["id"].map(all_predictions["XGBoost"])
    val_check["pred"] = (w * xgb_p + (1-w) * lgbm_p).clip(lower=0)

    idx = val_check.set_index(["store_nbr", "family"]).index
    val_check.loc[idx.isin(zero_force_combos), "pred"] = 0.0

    score = np.sqrt(mean_squared_log_error(val_check["sales"], val_check["pred"]))
    if best_weight_score is None or score < best_weight_score:
        best_weight_score = score
        best_weight = w
    print(f"weight XGBoost={w:.2f}: RMSLE = {score:.4f}")

print(f"\nBest weight: XGBoost={best_weight:.2f}, LightGBM={1-best_weight:.2f}, RMSLE={best_weight_score:.4f}")

In [ ]:
val_export = val_true.copy()
lgbm_p = val_export["id"].map(all_predictions["LightGBM"])
xgb_p = val_export["id"].map(all_predictions["XGBoost"])

val_export["lgbm_pred"] = lgbm_p
val_export["xgb_pred"] = xgb_p
val_export["ensemble_pred"] = (0.70 * xgb_p + 0.30 * lgbm_p).clip(lower=0)

idx = val_export.set_index(["store_nbr", "family"]).index
val_export.loc[idx.isin(zero_force_combos), "ensemble_pred"] = 0.0

val_export["abs_log_error"] = np.abs(
    np.log1p(val_export["ensemble_pred"]) - np.log1p(val_export["sales"])
)

val_export = val_export.sort_values("abs_log_error", ascending=False)

val_export.to_csv(f"{DATA_DIR}/validation_results.csv", index=False)
print("Saved to data/validation_results.csv")
print(val_export.shape)
print(val_export.head(10))

In [ ]:
X_train_full = train[feature_cols]
y_train_full = train["sales_log"]

X_train_full_encoded = X_train_full.copy()
for col in categorical_cols:
    X_train_full_encoded[col] = X_train_full_encoded[col].cat.codes

t0 = time.time()
final_lgbm = lgb.LGBMRegressor(n_estimators=486, **best_params)
final_lgbm.fit(X_train_full, y_train_full)
print(f"Final LightGBM trained in {time.time()-t0:.1f}s")

t0 = time.time()
final_xgb = xgb.XGBRegressor(
    n_estimators=486, learning_rate=0.0404, max_depth=10,
    min_child_weight=43, subsample=0.8245, colsample_bytree=0.9222,
    reg_alpha=0.0177, reg_lambda=2.2852, random_state=42, verbosity=0
)
final_xgb.fit(X_train_full_encoded, y_train_full)
print(f"Final XGBoost trained in {time.time()-t0:.1f}s")

In [ ]:
test = pd.read_parquet(f"{DATA_DIR}/test_processed_v3.parquet")
for col in categorical_cols:
    test[col] = test[col].astype("category")

print("Test shape:", test.shape)
history_real = train[["date", "store_nbr", "family", "sales", "onpromotion"]].copy()

static_cols_real = [c for c in feature_cols if c not in [
    "lag_7", "lag_14", "lag_28", "lag_364",
    "rolling_mean_7", "rolling_std_7", "rolling_mean_28", "rolling_std_28",
    "promo_typical", "promo_relative", "onpromotion", "store_nbr", "family"
]]
static_feats_real = test[static_cols_real + ["date", "store_nbr", "family", "id"]].drop_duplicates()
test_onpromo_real = test[["date", "store_nbr", "family", "onpromotion"]].copy()

working_real = pd.concat([
    history_real,
    test[["date", "store_nbr", "family"]].merge(test_onpromo_real, on=["date", "store_nbr", "family"], how="left").assign(sales=np.nan)
], axis=0, ignore_index=True)

real_test_dates = sorted(test["date"].unique())
lgbm_final_preds = {}
xgb_final_preds = {}

for current_date in real_test_dates:
    working_real = working_real.sort_values(["store_nbr", "family", "date"])

    for lag in [7, 14, 28, 364]:
        working_real[f"lag_{lag}"] = working_real.groupby(["store_nbr", "family"])["sales"].shift(lag)

    for window in [7, 28]:
        working_real[f"rolling_mean_{window}"] = (
            working_real.groupby(["store_nbr", "family"])["sales"]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        )
        working_real[f"rolling_std_{window}"] = (
            working_real.groupby(["store_nbr", "family"])["sales"]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=1).std())
        )

    working_real["promo_typical"] = (
        working_real.groupby(["store_nbr", "family"])["onpromotion"]
        .transform(lambda x: x.shift(1).rolling(90, min_periods=1).mean())
    )
    working_real["promo_relative"] = working_real["onpromotion"] / (working_real["promo_typical"] + 1)

    day_mask = (working_real["date"] == current_date) & (working_real["sales"].isna())
    day_rows = working_real[day_mask].copy()
    day_rows = day_rows.merge(static_feats_real, on=["date", "store_nbr", "family"], how="left")

    for col in categorical_cols:
        day_rows[col] = day_rows[col].astype("category")

    X_day_lgbm = day_rows[feature_cols]
    X_day_xgb = day_rows[feature_cols].copy()
    for col in categorical_cols:
        X_day_xgb[col] = X_day_xgb[col].cat.codes

    lgbm_pred_log = final_lgbm.predict(X_day_lgbm)
    xgb_pred_log = final_xgb.predict(X_day_xgb)

    lgbm_pred = np.clip(np.expm1(lgbm_pred_log), 0, None)
    xgb_pred = np.clip(np.expm1(xgb_pred_log), 0, None)

    ensemble_pred = 0.70 * xgb_pred + 0.30 * lgbm_pred

    idx = day_rows.set_index(["store_nbr", "family"]).index
    zero_mask = idx.isin(zero_force_combos)
    ensemble_pred[zero_mask] = 0.0

    for idx2, (_, row) in enumerate(day_rows.iterrows()):
        lgbm_final_preds[row["id"]] = lgbm_pred[idx2]
        xgb_final_preds[row["id"]] = xgb_pred[idx2]

    update_mask = (working_real["date"] == current_date)
    working_real.loc[update_mask, "sales"] = working_real.loc[update_mask].merge(
        day_rows[["store_nbr", "family", "date"]].assign(pred=ensemble_pred),
        on=["store_nbr", "family", "date"], how="left"
    )["pred"].values

    print(f"Predicted {current_date.date()}")

print("Done.")

In [ ]:
submission_final = pd.DataFrame({"id": list(lgbm_final_preds.keys())})
submission_final["lgbm"] = submission_final["id"].map(lgbm_final_preds)
submission_final["xgb"] = submission_final["id"].map(xgb_final_preds)
submission_final["sales"] = 0.70 * submission_final["xgb"] + 0.30 * submission_final["lgbm"]

id_to_combo = test.set_index("id")[["store_nbr", "family"]]
submission_final = submission_final.merge(id_to_combo, on="id", how="left")
idx = submission_final.set_index(["store_nbr", "family"]).index
submission_final.loc[idx.isin(zero_force_combos), "sales"] = 0.0

submission_final = submission_final[["id", "sales"]].sort_values("id").reset_index(drop=True)

print(submission_final.head())
print(submission_final.shape)
print(submission_final["sales"].describe())

submission_final.to_csv(f"{DATA_DIR}/submission_final.csv", index=False)
print("\nSaved to data/submission_final.csv")

In [ ]:
seeds = [42, 123, 777]
lgbm_models_multi = {}
xgb_models_multi = {}

for seed in seeds:
    t0 = time.time()
    params_seed = {**best_params, "random_state": seed}
    m = lgb.LGBMRegressor(n_estimators=486, **params_seed)
    m.fit(X_train_full, y_train_full)
    lgbm_models_multi[seed] = m
    print(f"LightGBM seed={seed} trained in {time.time()-t0:.1f}s")

    t0 = time.time()
    m2 = xgb.XGBRegressor(
        n_estimators=486, learning_rate=0.0404, max_depth=10,
        min_child_weight=43, subsample=0.8245, colsample_bytree=0.9222,
        reg_alpha=0.0177, reg_lambda=2.2852, random_state=seed, verbosity=0
    )
    m2.fit(X_train_full_encoded, y_train_full)
    xgb_models_multi[seed] = m2
    print(f"XGBoost seed={seed} trained in {time.time()-t0:.1f}s")

print("\nAll seed models trained.")

In [ ]:
seeds = [42, 123, 777]
lgbm_honest_multi = {}
xgb_honest_multi = {}

for seed in seeds:
    params_seed = {**best_params, "random_state": seed}
    m = lgb.LGBMRegressor(n_estimators=486, **params_seed)
    m.fit(X_train, y_train)
    lgbm_honest_multi[seed] = m

    m2 = xgb.XGBRegressor(
        n_estimators=486, learning_rate=0.0404, max_depth=10,
        min_child_weight=43, subsample=0.8245, colsample_bytree=0.9222,
        reg_alpha=0.0177, reg_lambda=2.2852, random_state=seed, verbosity=0
    )
    m2.fit(X_train_encoded, y_train)
    xgb_honest_multi[seed] = m2

    print(f"Seed {seed} honest models trained.")

print("Done.")

In [ ]:
history_val = train_honest[["date", "store_nbr", "family", "sales", "onpromotion"]].copy()
test_onpromo = train[train["date"] >= val_start][["date", "store_nbr", "family", "onpromotion"]].copy()

static_cols_val = [c for c in feature_cols if c not in [
    "lag_7", "lag_14", "lag_28", "lag_364",
    "rolling_mean_7", "rolling_std_7", "rolling_mean_28", "rolling_std_28",
    "promo_typical", "promo_relative", "onpromotion", "store_nbr", "family"
]]
static_features_val = train[train["date"] >= val_start][
    static_cols_val + ["date", "store_nbr", "family", "id"]
].drop_duplicates()

print("Setup done.")

In [ ]:
history_val = train_honest[["date", "store_nbr", "family", "sales", "onpromotion"]].copy()
test_onpromo = train[train["date"] >= val_start][["date", "store_nbr", "family", "onpromotion"]].copy()
val_dates = sorted(val_true["date"].unique())

static_cols_val = [c for c in feature_cols if c not in [
    "lag_7", "lag_14", "lag_28", "lag_364",
    "rolling_mean_7", "rolling_std_7", "rolling_mean_28", "rolling_std_28",
    "promo_typical", "promo_relative", "onpromotion", "store_nbr", "family"
]]
static_features_val = train[train["date"] >= val_start][
    static_cols_val + ["date", "store_nbr", "family", "id"]
].drop_duplicates()

print("Setup done.")

In [ ]:
working_val3 = pd.concat([
    history_val,
    val_true[["date", "store_nbr", "family"]].merge(test_onpromo, on=["date", "store_nbr", "family"], how="left").assign(sales=np.nan)
], axis=0, ignore_index=True)

multi_seed_preds = {}

for current_date in val_dates:
    working_val3 = working_val3.sort_values(["store_nbr", "family", "date"])

    for lag in [7, 14, 28, 364]:
        working_val3[f"lag_{lag}"] = working_val3.groupby(["store_nbr", "family"])["sales"].shift(lag)

    for window in [7, 28]:
        working_val3[f"rolling_mean_{window}"] = (
            working_val3.groupby(["store_nbr", "family"])["sales"]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        )
        working_val3[f"rolling_std_{window}"] = (
            working_val3.groupby(["store_nbr", "family"])["sales"]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=1).std())
        )

    working_val3["promo_typical"] = (
        working_val3.groupby(["store_nbr", "family"])["onpromotion"]
        .transform(lambda x: x.shift(1).rolling(90, min_periods=1).mean())
    )
    working_val3["promo_relative"] = working_val3["onpromotion"] / (working_val3["promo_typical"] + 1)

    day_mask = (working_val3["date"] == current_date) & (working_val3["sales"].isna())
    day_rows = working_val3[day_mask].copy()
    day_rows = day_rows.merge(static_features_val.drop(columns=["onpromotion"], errors="ignore"), on=["date", "store_nbr", "family"], how="left")

    for col in categorical_cols:
        day_rows[col] = day_rows[col].astype("category")

    X_day_lgbm = day_rows[feature_cols]
    X_day_xgb = day_rows[feature_cols].copy()
    for col in categorical_cols:
        X_day_xgb[col] = X_day_xgb[col].cat.codes

    lgbm_preds_seeds = np.mean([np.expm1(lgbm_honest_multi[s].predict(X_day_lgbm)) for s in seeds], axis=0)
    xgb_preds_seeds = np.mean([np.expm1(xgb_honest_multi[s].predict(X_day_xgb)) for s in seeds], axis=0)

    lgbm_preds_seeds = np.clip(lgbm_preds_seeds, 0, None)
    xgb_preds_seeds = np.clip(xgb_preds_seeds, 0, None)

    ensemble_pred = 0.70 * xgb_preds_seeds + 0.30 * lgbm_preds_seeds

    idx = day_rows.set_index(["store_nbr", "family"]).index
    ensemble_pred[idx.isin(zero_force_combos)] = 0.0

    for idx2, (_, row) in enumerate(day_rows.iterrows()):
        multi_seed_preds[row["id"]] = ensemble_pred[idx2]

    update_mask = (working_val3["date"] == current_date)
    working_val3.loc[update_mask, "sales"] = working_val3.loc[update_mask].merge(
        day_rows[["store_nbr", "family", "date"]].assign(pred=ensemble_pred),
        on=["store_nbr", "family", "date"], how="left"
    )["pred"].values

    print(f"Predicted {current_date.date()}")

val_check_multi = val_true.copy()
val_check_multi["pred"] = val_check_multi["id"].map(multi_seed_preds)
score_multi = np.sqrt(mean_squared_log_error(val_check_multi["sales"], val_check_multi["pred"]))
print(f"\nMulti-seed ensemble RMSLE: {score_multi:.4f}")
print(f"Previous best (single seed): 0.3996")

In [ ]:
working_real2 = pd.concat([
    history_real,
    test[["date", "store_nbr", "family"]].merge(test_onpromo_real, on=["date", "store_nbr", "family"], how="left").assign(sales=np.nan)
], axis=0, ignore_index=True)

multi_seed_final_preds = {}

for current_date in real_test_dates:
    working_real2 = working_real2.sort_values(["store_nbr", "family", "date"])

    for lag in [7, 14, 28, 364]:
        working_real2[f"lag_{lag}"] = working_real2.groupby(["store_nbr", "family"])["sales"].shift(lag)

    for window in [7, 28]:
        working_real2[f"rolling_mean_{window}"] = (
            working_real2.groupby(["store_nbr", "family"])["sales"]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        )
        working_real2[f"rolling_std_{window}"] = (
            working_real2.groupby(["store_nbr", "family"])["sales"]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=1).std())
        )

    working_real2["promo_typical"] = (
        working_real2.groupby(["store_nbr", "family"])["onpromotion"]
        .transform(lambda x: x.shift(1).rolling(90, min_periods=1).mean())
    )
    working_real2["promo_relative"] = working_real2["onpromotion"] / (working_real2["promo_typical"] + 1)

    day_mask = (working_real2["date"] == current_date) & (working_real2["sales"].isna())
    day_rows = working_real2[day_mask].copy()
    day_rows = day_rows.merge(static_feats_real, on=["date", "store_nbr", "family"], how="left")

    for col in categorical_cols:
        day_rows[col] = day_rows[col].astype("category")

    X_day_lgbm = day_rows[feature_cols]
    X_day_xgb = day_rows[feature_cols].copy()
    for col in categorical_cols:
        X_day_xgb[col] = X_day_xgb[col].cat.codes

    lgbm_preds_seeds = np.mean([np.expm1(lgbm_models_multi[s].predict(X_day_lgbm)) for s in seeds], axis=0)
    xgb_preds_seeds = np.mean([np.expm1(xgb_models_multi[s].predict(X_day_xgb)) for s in seeds], axis=0)

    lgbm_preds_seeds = np.clip(lgbm_preds_seeds, 0, None)
    xgb_preds_seeds = np.clip(xgb_preds_seeds, 0, None)

    ensemble_pred = 0.70 * xgb_preds_seeds + 0.30 * lgbm_preds_seeds

    idx = day_rows.set_index(["store_nbr", "family"]).index
    ensemble_pred[idx.isin(zero_force_combos)] = 0.0

    for idx2, (_, row) in enumerate(day_rows.iterrows()):
        multi_seed_final_preds[row["id"]] = ensemble_pred[idx2]

    update_mask = (working_real2["date"] == current_date)
    working_real2.loc[update_mask, "sales"] = working_real2.loc[update_mask].merge(
        day_rows[["store_nbr", "family", "date"]].assign(pred=ensemble_pred),
        on=["store_nbr", "family", "date"], how="left"
    )["pred"].values

    print(f"Predicted {current_date.date()}")

print("Done.")

In [ ]:
submission_multi = pd.DataFrame({"id": list(multi_seed_final_preds.keys())})
submission_multi["sales"] = submission_multi["id"].map(multi_seed_final_preds)
submission_multi = submission_multi.sort_values("id").reset_index(drop=True)

print(submission_multi.head())
print(submission_multi.shape)
print(submission_multi["sales"].describe())

submission_multi.to_csv(f"{DATA_DIR}/submission_multiseed.csv", index=False)
print("\nSaved to data/submission_multiseed.csv")